# DVFM Latent-Input Diagnostic

This notebook compares one trained original DVFM decoder under five conditions:

- posterior mean \(q_\phi(z\mid x,t,\delta)\);
- oracle true frailty mapped into the learned raw latent coordinate;
- aggregate-posterior marginalization using common draws;
- fixed \(z=0\);
- separately trained no-latent reference.

The posterior and true-frailty modes are diagnostics, not valid baseline predictions.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(
    "outputs/support_clayton_frailty_tau05_latent_prediction_diagnostics"
)
results = json.loads((OUTPUT_DIR / "results.json").read_text())
diagnostics = results["latent_prediction_diagnostics"]
predictions = pd.read_csv(
    OUTPUT_DIR / "latent_prediction_diagnostic_predictions.csv"
)
brier = pd.read_csv(
    OUTPUT_DIR / "latent_prediction_diagnostic_brier.csv"
)

summary = (
    pd.DataFrame(diagnostics["all_test"])
    .T
    .reset_index(names="mode")
)
summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric, title in [
    (axes[0], "oracle_ci", "Oracle CI ↑"),
    (axes[1], "oracle_ibs", "Oracle IBS ↓"),
    (axes[2], "median_time_mae", "Median-time MAE ↓"),
]:
    ax.bar(summary["mode"], summary[metric])
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for column in brier.columns:
    if column.endswith("_oracle_brier"):
        ax.plot(
            brier["time"],
            brier[column],
            label=column.replace("_oracle_brier", ""),
        )
ax.set_xlabel("Time")
ax.set_ylabel("Oracle Brier score")
ax.set_title("Where each latent-input mode gains or loses calibration")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Interpretation

- **Posterior and true latent good; aggregate poor:** test-time marginalization is the bottleneck.
- **\(z=0\) poor:** the decoder has failed to preserve a strong covariate-only pathway.
- **True latent poor:** the decoder itself is not useful even when given the right dependence state.
- **Aggregate close to \(z=0\):** latent effects mostly cancel under population marginalization.


In [ ]:
diagnostics["interpretation_flags"]
